← [P3 · How we observe](p3_how_we_observe.ipynb) · [Index](../README.md) · [00 · Light and light curves](../00_light_and_light_curves.ipynb) →
<!--nav-->

# P4 · What the data actually looks like

The last gap between "I understand the concepts" and "I can work with this". Astronomical
data comes in four shapes, almost always inside one file format, and carries two conventions
that cause more errors than anything else.

**You'll learn:** the FITS format · images, light curves, spectra and catalogues, each with a
real example · world coordinate systems · and the time conventions that will silently ruin
your results.

## 1. FITS: the format everything arrives in

**FITS** (Flexible Image Transport System) has been the standard since the late 1970s and is
still universal. Its design is why it survived:

- A file is a list of **HDUs** (Header/Data Units).
- Each HDU has a **header** of human-readable `KEYWORD = value / comment` lines, then a data
  block — an N-dimensional array or a table.
- **The header travels with the data.** Instrument, exposure time, coordinates, units,
  processing history all live in the file. There is no separate metadata to lose.

That last property is the whole point. A FITS file from 1985 is still readable and still
self-describing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from skyplay import data, plotting

plotting.use_style()

# A real Kepler target pixel file: images of Kepler-8 over one quarter.
tpf = data.load_tpf('kepler-8', quarter=4)

print('HDUs in this file:')
for i, hdu in enumerate(tpf.hdu):
    kind = type(hdu).__name__
    shape = getattr(hdu.data, 'shape', None)
    print(f'  [{i}] {kind:18s} {str(shape):>22s}  {hdu.header.get("EXTNAME", "PRIMARY")}')

In [ ]:
# The header: metadata that shipped with the pixels.
header = tpf.hdu[0].header
for key in ('TELESCOP', 'INSTRUME', 'OBJECT', 'QUARTER', 'RA_OBJ', 'DEC_OBJ',
            'KEPMAG', 'DATE-OBS'):
    if key in header:
        print(f'  {key:10s} = {header[key]}')

## 2. Data shape 1 — an image

The most literal product: a 2D array where each value is the brightness of one pixel. A
Kepler target pixel file is a *stack* of small images over time, so it's really 3D
(time, row, column).

In [ ]:
frame = tpf.flux[100].value

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
im = axes[0].imshow(frame, origin='lower', cmap=plotting.SEQUENTIAL)
axes[0].set_title('One frame: raw pixel values')
axes[0].set_xlabel('column'); axes[0].set_ylabel('row')
fig.colorbar(im, ax=axes[0], label='electrons / second')

# Log scaling, because astronomical brightness spans orders of magnitude.
im2 = axes[1].imshow(np.log10(np.clip(frame, 1, None)), origin='lower',
                     cmap=plotting.SEQUENTIAL)
axes[1].set_title('Same frame, log scale — the faint wings appear')
axes[1].set_xlabel('column')
fig.colorbar(im2, ax=axes[1], label='log10(e-/s)')
plt.show()

print(f'shape (time, row, col): {tpf.flux.shape}')
print('The star is the bright blob. It spreads over several pixels because a telescope')
print('is not perfectly sharp -- that spread is the point spread function (PSF).')

### World coordinates

Pixel (3, 4) is meaningless to anyone else. A **WCS** in the header maps pixel positions to
sky coordinates (right ascension and declination), so any two instruments can be compared.

In [ ]:
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

wcs = WCS(tpf.hdu[2].header)
centre = wcs.pixel_to_world(frame.shape[1] // 2, frame.shape[0] // 2)
print(f'centre pixel -> RA {centre.ra.deg:.5f} deg, Dec {centre.dec.deg:.5f} deg')
print(f'             -> {centre.to_string("hmsdms")}')

# Use proj_plane_pixel_scales, not a single matrix element: the CD matrix encodes
# rotation as well as scale, so reading CD1_1 alone under-reports it (2.52 vs 3.97 here).
scale_x, scale_y = proj_plane_pixel_scales(wcs) * 3600
print(f'\npixel scale: {scale_x:.2f} x {scale_y:.2f} arcsec/pixel')
print('Kepler pixels are 3.98 arcsec, so that checks out -- always verify a derived')
print('number against a known instrument value when you can.')

## 3. Data shape 2 — a light curve

Sum the pixels in an aperture, once per frame, and you have brightness against time. This is
the product the whole repo is built on, and notebook 02 covers how the summing works.

In [ ]:
lc = data.load_stitched('kepler-8')

print(f'{len(lc)} measurements over {(lc.time.max() - lc.time.min()).value:.0f} days')
print(f'columns: time, flux, flux_err   (+ metadata in lc.meta)')
print(f'time format: {lc.time.format}, scale: {lc.time.scale}')
print()
print(lc[:4])

lc.scatter(s=1)
plt.show();

## 4. Data shape 3 — a spectrum

Brightness against **wavelength** instead of time. Where composition, temperature and
velocity come from. Absorption lines appear at wavelengths characteristic of specific atoms,
so a spectrum is a chemical fingerprint — and if the whole pattern is shifted, that shift is
a velocity (Doppler), which is how radial-velocity planet masses are measured.

Here's a real galaxy spectrum from SDSS.

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.sdss import SDSS

pos = SkyCoord('0h8m05.63s +14d50m23.3s', frame='icrs')
xid = SDSS.query_region(pos, spectro=True, radius=5 * u.arcsec)
spectra = SDSS.get_spectra(matches=xid)

spec = spectra[0][1].data          # the spectrum lives in HDU 1
wavelength = 10 ** spec['loglam']  # SDSS stores log10(wavelength / Angstrom)
flux = spec['flux']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wavelength, flux, lw=0.7, color=plotting.SERIES[0])
ax.set_xlabel(r'Wavelength ($\AA$)')
ax.set_ylabel('Flux')
ax.set_title('A real SDSS spectrum — every wiggle is physics, not noise')
plt.show()

print('Peaks are emission lines, dips are absorption. Their identity tells you which')
print('atoms are present; their shift tells you how fast the object is moving.')

## 5. Data shape 4 — a catalogue

Most astronomy is *tables*: one row per object, columns of measured properties. Gaia's is 2
billion rows. You query them rather than download them, and `astroquery` speaks to most of
the major archives.

In [ ]:
from astroquery.gaia import Gaia

job = Gaia.launch_job('''
    SELECT TOP 5 source_id, ra, dec, parallax, phot_g_mean_mag, bp_rp
    FROM gaiadr3.gaia_source
    WHERE parallax > 100
    ORDER BY phot_g_mean_mag ASC
''')
nearby = job.get_results()
print('The 5 brightest stars within 10 pc, from Gaia DR3:')
print(nearby)

print()
print(f'distance to the first = {1000 / nearby["parallax"][0]:.2f} pc')

## 6. The two conventions that cause real errors

### Units

Astropy attaches units to numbers, and this is not decoration — it catches whole classes of
mistake at runtime rather than in your conclusions.

In [ ]:
from astropy import constants as const

# Units convert, and refuse to do nonsense.
distance = 1070 * u.pc
print(f'{distance} = {distance.to(u.lightyear):.3g} = {distance.to(u.km):.3g}')

# Combine them and the units follow along.
light_time = (distance / const.c).to(u.yr)
print(f'light travel time: {light_time:.0f}')

try:
    _ = 5 * u.m + 3 * u.kg
except u.UnitConversionError as exc:
    print(f'\nadding metres to kilograms -> {type(exc).__name__}')
    print('   ...which is exactly what you want to happen.')

### Time — the one that will actually bite you

Astronomical timestamps carry two independent choices, and getting either wrong produces
answers that look plausible and are wrong.

**The zero point.** Missions subtract a big constant so the numbers stay small:

| Format | Offset from Julian Date | Used by |
|---|---|---|
| **JD** | 0 | General |
| **BJD** | 0 | Barycentric JD |
| **BKJD** | BJD − 2454833 | **Kepler** |
| **BTJD** | BJD − 2457000 | **TESS** |

If you ever compare an epoch from a paper against one you measured and are wrong by
~2.4 million, this is why.

**The scale.** `UTC` has leap seconds; `TDB` does not. Mission timestamps are on `TDB`.

**"Barycentric" matters.** Times are corrected to the Solar System barycentre rather than
Earth, because Earth's orbit moves us ±8 light-minutes relative to the Sun over a year.
Uncorrected, a perfectly regular transit would appear to drift by up to ~16 minutes annually.

Let astropy handle all of it. Never subtract raw floats from different missions.

In [ ]:
from astropy.time import Time

# Kepler-8 b's transit epoch, as this repo carries it (BKJD).
t_kepler = Time(131.6930, format='bkjd', scale='tdb')

print(f'BKJD        : {t_kepler.value:.4f}')
print(f'JD          : {t_kepler.jd:.4f}')
print(f'ISO (UTC)   : {t_kepler.utc.iso}')
print(f'as BTJD     : {t_kepler.btjd:.4f}   <- what TESS would call the same instant')
print()
print(f'BKJD -> BTJD offset: {t_kepler.jd - 2457000 - t_kepler.value:.0f} days')
print('Getting that wrong is a ~6 year error. It happens.')

## Recap

- **FITS** carries data *and* its metadata in one self-describing file.
- Four shapes: **image** (pixels), **light curve** (time), **spectrum** (wavelength),
  **catalogue** (rows). Nearly all analysis is one of these.
- **WCS** turns pixels into sky coordinates so instruments can be compared.
- **Units and time systems are load-bearing.** Use `astropy.units` and `astropy.time` and
  the most common silent errors become impossible.

You now have the vocabulary for the main learning path. Start at
[`00_light_and_light_curves.ipynb`](../00_light_and_light_curves.ipynb).

## Learning resources
- 📘 [Astropy documentation](https://docs.astropy.org/) — units, time, coordinates, FITS, WCS
- 📗 [astroquery](https://astroquery.readthedocs.io/) — one interface to most archives
- 🏛️ [MAST](https://archive.stsci.edu/) · [Gaia archive](https://gea.esac.esa.int/archive/) · [SDSS](https://www.sdss.org/)
- 🌍 [FITS standard](https://fits.gsfc.nasa.gov/fits_standard.html)